In [8]:
import sys
sys.path.append("../scripts")

In [9]:
import numpy as np
import pandas as pd
import preprocessing
data = pd.read_csv('../data/raw/car_price_prediction.csv')
data.head()

,ID,Price,Levy,Manufacturer,Model,Prod. year,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Doors,Wheel,Color,Airbags
0,45654403,13328,1399,LEXUS,RX 450,2010,Jeep,Yes,Hybrid,3.5,186005 km,6.0,Automatic,4x4,04-May,Left wheel,Silver,12
1,44731507,16621,1018,CHEVROLET,Equinox,2011,Jeep,No,Petrol,3,192000 km,6.0,Tiptronic,4x4,04-May,Left wheel,Black,8
2,45774419,8467,-,HONDA,FIT,2006,Hatchback,No,Petrol,1.3,200000 km,4.0,Variator,Front,04-May,Right-hand drive,Black,2
3,45769185,3607,862,FORD,Escape,2011,Jeep,Yes,Hybrid,2.5,168966 km,4.0,Automatic,4x4,04-May,Left wheel,White,0
4,45809263,11726,446,HONDA,FIT,2014,Hatchback,Yes,Petrol,1.3,91901 km,4.0,Automatic,Front,04-May,Left wheel,Silver,4


In [10]:
data = preprocessing.preprocessing_pipeline(data)

Preprocessing started...
Initial shape: (19237, 18)
After dropping duplicates: (18924, 18)
Replacing categorical values...
After cleaning outliers: (16037, 18)
Feature engineering...
Drooping columns...
Final shape:  (16037, 16)


In [11]:
data.head()

,Price,Levy,Manufacturer,Model,Category,Leather interior,Fuel type,Engine volume,Mileage,Cylinders,Gear box type,Drive wheels,Wheel,Color,Airbags,Age
0,13328,1399,LEXUS,RX 450,Jeep,Yes,Hybrid,3.5,186005,6.0,Automatic,4x4,Left wheel,Silver,12,16
1,16621,1018,CHEVROLET,Equinox,Jeep,No,Petrol,3.0,192000,6.0,Tiptronic,4x4,Left wheel,Black,8,15
2,8467,0,HONDA,FIT,Hatchback,No,Petrol,1.3,200000,4.0,Variator,Front,Right-hand drive,Black,2,20
3,3607,862,FORD,Escape,Jeep,Yes,Hybrid,2.5,168966,4.0,Automatic,4x4,Left wheel,White,0,15
4,11726,446,HONDA,FIT,Hatchback,Yes,Petrol,1.3,91901,4.0,Automatic,Front,Left wheel,Silver,4,12


In [12]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
one_hot_columns = ['Leather interior', 'Gear box type', 'Drive wheels', 'Wheel']

oh_encoder = OneHotEncoder(sparse_output=False,handle_unknown='ignore')
oh_encoded_train = oh_encoder.fit_transform(data[one_hot_columns])
oh_encoded_columns = oh_encoder.get_feature_names_out(one_hot_columns)

In [13]:
oh_encoded_train_df = pd.DataFrame(oh_encoded_train, columns=oh_encoded_columns, index=data.index)


In [14]:
# to concatenate the original data

data = pd.concat([data, oh_encoded_train_df], axis=1)
data.drop(columns=one_hot_columns, inplace=True) 

In [15]:
import pickle  # to save the one hot encoder model
with open('../models/one_hot_encoder.pkl', 'wb') as f:
    pickle.dump(oh_encoder, f)

In [16]:
label_encode_columns = ['Manufacturer', 'Model', 'Category', 'Fuel type','Color']

label_encoders = {}
for column in label_encode_columns:
    label_encoder = LabelEncoder()
    data[column] = label_encoder.fit_transform(data[column])
    label_encoders[column] = label_encoder


In [17]:
with open('../models/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

In [18]:
x = data.drop('Price', axis=1)
y = data['Price']

In [19]:
# i remove validation test
from sklearn.model_selection import train_test_split
x_train, X_test , y_train , y_test = train_test_split(x, y, train_size = 0.15, random_state = 42)

print(f"Train set: {len(X_test)} samples")
print(f"Test set: {len(x_train)} samples")


Train set: 13632 samples
Test set: 2405 samples


In [20]:
from sklearn.preprocessing import StandardScaler
numerical_columns = ['Levy', 'Engine volume', 'Mileage', 'Age']
scaler = StandardScaler()

x_train[numerical_columns] = scaler.fit_transform(x_train[numerical_columns])

X_test[numerical_columns] = scaler.transform(X_test[numerical_columns])

In [21]:
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [22]:
x_train

,Levy,Manufacturer,Model,Category,Fuel type,Engine volume,Mileage,Cylinders,Color,Airbags,...,Leather interior_Yes,Gear box type_Automatic,Gear box type_Manual,Gear box type_Tiptronic,Gear box type_Variator,Drive wheels_4x4,Drive wheels_Front,Drive wheels_Rear,Wheel_Left wheel,Wheel_Right-hand drive
8711,-1.254491,35,927,4,1,0.606308,1.910529,6.0,1,2,...,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0
615,0.352304,6,911,4,1,-0.926636,-0.036512,4.0,14,4,...,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1051,1.411028,3,89,1,5,1.457943,-1.624358,6.0,14,12,...,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
10290,-1.254491,35,412,3,5,-1.437617,0.452388,4.0,13,4,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
27,-1.254491,53,965,9,2,-1.096963,0.509199,4.0,14,8,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16106,1.547495,50,1024,4,1,0.946962,-0.652062,4.0,12,4,...,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
6437,0.429342,37,2,9,5,-0.245328,-0.735649,4.0,14,12,...,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1011,-1.254491,53,750,3,5,-0.585982,-0.374523,4.0,14,0,...,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
18952,1.325185,25,1121,4,5,2.309579,-0.080079,6.0,7,10,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0


In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

rf = RandomForestRegressor()
rf.fit(x_train, y_train)
y_val_pred = rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_val_pred))
r2 = r2_score(y_test, y_val_pred)

print(f"Root Mean Squared Error: {rmse}")
print(f"R^2 Score : {r2}")

Root Mean Squared Error: 6216.286686526353
R^2 Score : 0.6967670758443869


In [24]:
with open('../models/rf_model.pkl', 'wb') as f:
    pickle.dump(rf, f)